# 01 — Source Discovery

Live queries against data.sf.gov, run before any data is pulled, to confirm the sources named in `PROJECT_BRIEF.md` actually look the way the brief describes — nothing here is taken on faith. Full writeup: `docs/source_map.md`. Decisions made from these findings (what got cut and why): `docs/decision_log.md`.


In [1]:
import requests
import pandas as pd

BASE = "https://data.sf.gov"


## S1 — Fire Dept & EMS Dispatched Calls for Service (`nuek-vuh3`)


In [2]:
meta = requests.get(f"{BASE}/api/views/nuek-vuh3.json", timeout=30).json()
print(meta["name"])
print(meta["description"][:300])
print("columns:", len(meta["columns"]))
pd.DataFrame([{"field": c["fieldName"], "type": c["dataTypeName"]} for c in meta["columns"]]).head(15)


Fire Department and Emergency Medical Services Dispatched Calls for Service
<strong>A. SUMMARY</strong>
Fire Calls-For-Service includes all fire units' responses to 911 calls from the city's Computer-Aided Dispatch (“CAD”) system. This includes responses to Medical Incidents requiring EMS staff. Each record includes the call number, incident number, address, unit identifier
columns: 37


,field,type
0,call_number,text
1,unit_id,text
2,incident_number,text
3,call_type,text
4,call_date,calendar_date
5,watch_date,calendar_date
6,received_dttm,calendar_date
7,entry_dttm,calendar_date
8,dispatch_dttm,calendar_date
9,response_dttm,calendar_date


Confirms the dataset's own description covers **Fire and EMS** dispatches together, not EMS alone — this becomes relevant later when scoping the KPI to `call_type_group = 'Potentially Life-Threatening'` (`docs/assumptions.md` §3).


In [3]:
sample = requests.get(f"{BASE}/resource/nuek-vuh3.json", params={"$limit": 2, "$order": "received_dttm DESC"}, timeout=30).json()
pd.DataFrame(sample)[["call_number", "unit_id", "unit_type", "original_priority", "final_priority", "received_dttm", "dispatch_dttm"]]


,call_number,unit_id,unit_type,original_priority,final_priority,received_dttm,dispatch_dttm
0,262660269,AM122,PRIVATE,2,2,2026-09-23T03:19:26.000,2026-09-23T03:23:56.000
1,262660269,M503,MEDIC,2,2,2026-09-23T03:19:26.000,2026-09-23T03:21:45.000


A live sample confirms the grain directly: two rows can share the same `call_number` with different `unit_id`/`unit_type` - one row per unit dispatched to a call, exactly as `docs/source_map.md` describes.


## S3 — City Performance Scorecard Measures (`kc49-udxn`), measure 973


In [4]:
# actual IS NOT NULL: Socrata omits the key entirely (not null) for months with no
# reported value yet - the most recent months, given the ~3-month reporting lag.
rows = requests.get(f"{BASE}/resource/kc49-udxn.json",
                    params={"measure_code": "973", "$where": "actual IS NOT NULL",
                            "$order": "calendar_month DESC", "$limit": 6},
                    timeout=30).json()
pd.DataFrame(rows)[["measure_title", "calendar_month", "actual", "target"]]


,measure_title,calendar_month,actual,target
0,Percentage of ambulances that arrive on-scene ...,2026-06-30T00:00:00.000,0.884,0.9
1,Percentage of ambulances that arrive on-scene ...,2026-05-31T00:00:00.000,0.868,0.9
2,Percentage of ambulances that arrive on-scene ...,2026-04-30T00:00:00.000,0.885,0.9
3,Percentage of ambulances that arrive on-scene ...,2026-03-31T00:00:00.000,0.875,0.9
4,Percentage of ambulances that arrive on-scene ...,2026-02-28T00:00:00.000,0.873,0.9
5,Percentage of ambulances that arrive on-scene ...,2026-01-31T00:00:00.000,0.863,0.9


Confirms live: `measure_title` = "Percentage of ambulances that arrive on-scene within 10 minutes to life-threatening medical emergencies", `target` = 0.9, and June 2026's `actual` = 0.884 - matching the brief's headline figure exactly, pulled fresh rather than copied. Also confirms the scorecard's ~3-month reporting lag: no `actual` is reported yet for months after June 2026 as of this pull.


## S4 — Fire Incidents (`wr8u-xric`) — checked, then cut


In [5]:
meta4 = requests.get(f"{BASE}/api/views/wr8u-xric.json", timeout=30).json()
print(meta4["description"][:250])


<strong>Note 3/27/2024: This data pipeline was recently updated to improve data quality and efficiency. Incidents outside of SF city boundaries will no longer be assigned a supervisor_district and neighborhood_district. </strong>

Fire Incidents incl


The dataset's own metadata states it covers *"a summary of each **non-medical** incident."* It cannot cross-check or enrich EMS/ambulance calls - the two datasets don't overlap in subject matter. Cut with this evidence, not a hunch (`docs/decision_log.md`, 2026-09-23).


## Which measures exist for the Fire Department?


In [6]:
fd_measures = requests.get(f"{BASE}/resource/kc49-udxn.json",
                           params={"department": "Fire Department", "$select": "measure_code,measure_title", "$group": "measure_code,measure_title"},
                           timeout=30).json()
pd.DataFrame(fd_measures)


,measure_code,measure_title
0,973,Percentage of ambulances that arrive on-scene ...


**Exactly one** Fire Department scorecard measure exists (973). There is no official measure for call-processing time or hospital turnaround - M2 and M4 in this project have no official baseline to reconcile against; they are pipeline-derived only. Worth stating explicitly so `docs/decision_memo.md` doesn't imply an official comparator exists where none does.


## Summary

- S1 and S3 confirmed live, matching `PROJECT_BRIEF.md`'s figures exactly where checked (measure 973's June 2026 actual = 0.884).
- S4 confirmed as out of scope from its own metadata, not assumed.
- Fire Department has exactly one scorecard measure - flags a real reconciliation gap for M2/M4 addressed later in `notebooks/04_metrics_reconciliation.ipynb`.
- Full source map, ownership, grain, freshness, and every gap: `docs/source_map.md`.
